<a href="https://colab.research.google.com/github/devgomesai/TubeTalk-AI/blob/main/whisper_ai_yt_transcript.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# !pip install git+https://github.com/openai/whisper.git
# !sudo apt update && sudo apt install ffmpeg

In [5]:
# !whisper "/content/Dklv1DkYgK8.mp4" --model medium

In [7]:
!pip install yt_dlp

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 88.0 MB/s eta 0:00:00


In [57]:
from urllib.parse import urlparse, parse_qs
import yt_dlp
import whisper
import subprocess
import os

In [23]:
def get_video_id(url):
    parsed_url = urlparse(url)
    if parsed_url.hostname in ["www.youtube.com", "youtube.com"]:
        if parsed_url.path.startswith("/live/"):
            return parsed_url.path.split("/live/")[1]
        if parsed_url.path.startswith("/shorts/"):
          return parsed_url.path.split("/shorts/")[1].split("/")[0]
        return parse_qs(parsed_url.query).get("v", [None])[0]
    elif parsed_url.hostname == "youtu.be":
        return parsed_url.path[1:]
    return None

In [42]:
def download_audio(youtube_url, cookies_path="cookies.txt"):
    print("Downloading audio from YouTube for transcription...")
    try:
        video_id = get_video_id(youtube_url)
        if not video_id:
            print("Invalid YouTube URL")
            return "Invalid YouTube URL", None

        audio_dir = "audios"
        os.makedirs(audio_dir, exist_ok=True)

        ydl_opts = {
            "format": "bestaudio/best",
            "outtmpl": f"{audio_dir}/{video_id}.%(ext)s",
            "cookiefile": cookies_path  # <- THIS is the key part
        }

        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(youtube_url, download=True)
            print("Audio download complete")
            return f"{audio_dir}/{video_id}.{info['ext']}", info.get("title", "Unknown Title")

    except Exception as e:
        print(f"Audio download failed: {str(e)}")
        return f"Audio download failed: {str(e)}", None

In [54]:
youtube_url = str(input("Enter a youtube URL : "))

Enter a youtube URL :  https://www.youtube.com/watch?v=wjZofJX0v4M


In [55]:
path = download_audio(youtube_url, cookies_path="/content/cookies.txt")

[generic] Extracting URL:  https://www.youtube.com/watch?v=wjZofJX0v4M
[generic] watch?v=wjZofJX0v4M: Downloading webpage
[redirect] Following redirect to https://www.youtube.com/watch?v=wjZofJX0v4M
[youtube] Extracting URL: https://www.youtube.com/watch?v=wjZofJX0v4M
[youtube] wjZofJX0v4M: Downloading webpage
[youtube] wjZofJX0v4M: Downloading tv client config
[youtube] wjZofJX0v4M: Downloading tv player API JSON
[youtube] wjZofJX0v4M: Downloading web safari player API JSON
[youtube] wjZofJX0v4M: Downloading m3u8 information
[info] wjZofJX0v4M: Downloading 1 format(s): 251-7
[download] Sleeping 3.00 seconds as required by the site...
[download] Destination: audios/wjZofJX0v4M.webm
[download] 100% of   27.39MiB in 00:00:00 at 30.00MiB/s  
Audio download complete


In [56]:
print(str(path[0]))

audios/wjZofJX0v4M.webm


In [58]:
path_to_file = f"/content/{str(path[0])}"
subprocess.run(["whisper", path_to_file, "--model", "medium"])

CompletedProcess(args=['whisper', '/content/audios/wjZofJX0v4M.webm', '--model', 'medium'], returncode=0)